# 문제 1-1 : 기본 Chain 만들기 - AI 요리사

사용자가 입력한 재료로 만들 수 있는 요리를 추천합니다.

- `PromptTemplate` 으로 프롬프트 작성
- `ChatOpenAI` 모델 사용 (`get_llm()` 이 `ChatOpenAI` 인스턴스를 반환)
- `StrOutputParser` 로 결과를 문자열로 출력
- LCEL(`|`) 문법으로 체인 연결

### 공통 설정: API Key 로드 + LLM 생성 함수

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [2]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. PromptTemplate 으로 프롬프트 작성 ({ingredients} 가 사용자 입력 변수)
template_text = """당신은 친절한 AI 요리사입니다.
사용자가 가진 재료로 만들 수 있는 요리 1가지를 추천하고 간단한 레시피를 알려주세요.
반드시 아래 형식을 지켜 한국어로 답변하세요.

{ingredients}(으)로 만들 수 있는 요리를 추천드립니다!

추천 요리: (요리 이름)
재료: (필요한 재료)
조리법:
1. ...
2. ...
3. ...
"""

prompt_template = PromptTemplate.from_template(template_text)
print(prompt_template.input_variables)

['ingredients']


In [4]:
# 2. ChatOpenAI 모델 생성
llm = get_llm(temperature=0.7)

# 3. LCEL(|) 로 프롬프트 → 모델 → 문자열 파서 연결
chain = prompt_template | llm | StrOutputParser()

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [5]:
# 4. 실행: 예시 입력 "계란, 밥, 김치"
response = chain.invoke({"ingredients": "계란, 밥, 김치"})
print(type(response))  # str
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
계란, 밥, 김치(으)로 만들 수 있는 요리를 추천드립니다!

추천 요리: 김치볶음밥

재료:  
- 밥 1공기  
- 김치 ½컵 (잘게 썬 것)  
- 계란 1개  
- 식용유 2큰술  
- 간장 1작은술  
- 설탕 ½작은술  
- 대파 1줄기 (다진 것, 선택)  
- 참기름 ½작은술  
- 소금·후추 약간  

조리법:  
1. 팬에 식용유 1큰술을 두르고 중불에 달군 뒤, 다진 대파와 김치를 넣어 2~3분 정도 볶아 김치의 풍미를 끌어올린다.  
2. 밥을 넣고 김치와 잘 섞이도록 뒤섞으며 볶는다. 간장과 설탕을 넣고 간을 맞춘 뒤, 소금·후추로 마무리한다. 마지막에 참기름을 둘러 향을 더한다.  
3. 다른 팬에 식용유 1큰술을 두르고 약간의 물을 넣은 뒤 계란을 깨뜨려 스크램블하거나 반숙 계란 프라이를 만든다. 완성된 김치볶음밥 위에 올려서 즐긴다.  


In [6]:
# 사용자 입력을 받아서 실행 (빈 값이면 기본 재료 사용)
user_input = input("가지고 있는 재료를 입력하세요 (예: 토마토, 양파, 치즈): ").strip() or "토마토, 양파, 치즈"
print(f"입력: {user_input}\n")
print(chain.invoke({"ingredients": user_input}))

입력: 토마토, 양파, 치즈

토마토, 양파, 치즈(으)로 만들 수 있는 요리를 추천드립니다!

추천 요리: 치즈 토마토 양파 파스타  
재료: 스파게티 면 200g, 토마토 2개, 양파 1/2개, 모짜렐라 치즈 100g, 올리브오일 2큰술, 마늘 1쪽, 소금·후추 약간, 바질(선택) 약간  
조리법:  
1. 스파게티 면을 끓는 물에 소금을 넣고 알단테가 될 때까지 삶은 뒤 물기를 빼둡니다.  
2. 팬에 올리브오일을 두르고 다진 마늘과 채썬 양파를 넣어 투명해질 때까지 볶습니다. 여기에 깍둑썰기한 토마토를 넣고 3~4분간 끓여 소스가 살짝 졸아들게 합니다.  
3. 삶은 면을 팬에 넣고 소금·후추로 간을 맞춘 뒤, 위에 모짜렐라 치즈를 골고루 뿌리고 뚜껍을 덮어 치즈가 녹을 때까지 2분 정도 뜸을 들입니다. 마지막에 바질을 올려 완성합니다.
